# Lab 09: Relational Algebra with NYC Flight Data

**9 questions | 100 points | Fully autograded**

Practice relational algebra using real NYC flight data, then apply grouping to summarize airline activity.

- Q1–Q8: projection, selection, union, intersection, difference, cross-product, theta-join, and natural join — **82 points**.
- Q9: join, group, and count flights — **18 points**.

Implement the named functions for Q1–Q8. Q9 uses variables directly. Do not hard-code answers or modify input tables. For this simplified lab, work with pandas row semantics: repeated rows are retained. The union exercise uses concatenation (the duplicate-retaining variant, equivalent to SQL UNION ALL). Classical relational algebra uses set semantics. Pandas indexes are not relational attributes. Preserve requested column order. Tests compare row content rather than requiring a particular index.

Place `flights.csv`, `airlines.csv`, and `planes.csv` beside this notebook. All files are provided; no download is required while working.

Source: [nycflights13](https://nycflights13.tidyverse.org/), distributed as CSV through [Rdatasets](https://vincentarelbundock.github.io/Rdatasets/). Records describe 2013. The flight CSV retains all departure rows and eight original data columns.


<a id="question-navigation"></a>

## Question Navigation

- [Q1: Projection — 8 points](#q1)
- [Q2: Selection — 10 points](#q2)
- [Q3: Union (Keeping All Rows) — 10 points](#q3)
- [Q4: Intersection — 10 points](#q4)
- [Q5: Set Difference — 12 points](#q5)
- [Q6: Cross-product — 12 points](#q6)
- [Q7: Theta-Join — 10 points](#q7)
- [Q8: Natural Join — 10 points](#q8)
- [Q9: Flights per Airline — 18 points](#q9)


## Lab Setup

In [ ]:
import numpy as np
import pandas as pd

import otter
grader = otter.Notebook()

In [ ]:
flights = pd.read_csv('data/flights.csv')
airlines = pd.read_csv('data/airlines.csv')
planes = pd.read_csv('data/planes.csv')

# One-column destination tables with identical schemas.
jfk_routes = pd.DataFrame({'dest': pd.unique(flights.loc[flights['origin'].eq('JFK'), 'dest'])})
lga_routes = pd.DataFrame({'dest': pd.unique(flights.loc[flights['origin'].eq('LGA'), 'dest'])})

# Small real-data subsets keep pairwise operations manageable.
small_airlines = airlines[['carrier', 'name']].head(3)
origins = pd.DataFrame({'origin': sorted(flights['origin'].unique())})
plane_sample = planes.loc[planes['year'].notna(), ['tailnum', 'year']].head(5)
flights.head()

In [ ]:
airlines.head()

In [ ]:
plane_sample

<a id="q1"></a>

## Question 1: Projection — 8 Points

Implement `project(df, columns)` to select the requested columns, in the requested order. Keep all input rows.

Example: projecting two records onto `['carrier']` returns two rows, even if their carrier codes are the same.

Apply it to obtain origin–destination columns from `flights`.

[Return to outline](#question-navigation)


In [ ]:
def project(df, columns):
    return ...

routes = project(flights, ['origin', 'dest'])
routes.head()

In [ ]:
grader.check("q1")

<a id="q2"></a>

## Question 2: Selection — 10 Points

Implement `select(df, condition)`. The condition is a function applied to **each row** and returns `True` or `False`. Return only the matching rows, retaining all columns.

Example: `select(flights, lambda row: row['origin'] == 'JFK')` returns JFK departures.

Use the example to create `jfk_flights`.

[Return to outline](#question-navigation)


In [ ]:
def select(df, condition):
    mask = ... 
    return ...

jfk_flights = select(flights, lambda row: row['origin'] == 'JFK')
jfk_flights.head()

In [ ]:
grader.check("q2")

<a id="q3"></a>

## Question 3: Union (Keeping All Rows) — 10 Points

Implement `union(df1, df2)` for tables with identical column names, order, and compatible types. Concatenate the two tables, keeping all rows, including repeated rows. Keep the original column order. This is the duplicate-retaining variant (SQL UNION ALL), rather than classical set union.

Use it to create `either_destinations`: destinations served from JFK **or** LGA. Example: combining `[BOS, LAX]` and `[BOS, ATL]` gives `[BOS, LAX, BOS, ATL]`.

[Return to outline](#question-navigation)


In [ ]:
def union(df1, df2):
    return ...

either_destinations = union(jfk_routes, lga_routes)
either_destinations.head()

In [ ]:
grader.check("q3")

<a id="q4"></a>

## Question 4: Intersection — 10 Points

Implement `intersection(df1, df2)` for identical schemas. Match **complete rows** occurring in both tables using an inner merge on all columns. You do not need to remove repeated rows.

Create `shared_destinations`: destinations served from both JFK and LGA. Example: `[BOS, LAX] ∩ [BOS, ATL]` gives `[BOS]`.

[Return to outline](#question-navigation)


In [ ]:
def intersection(df1, df2):
    return ...

shared_destinations = intersection(jfk_routes, lga_routes)
shared_destinations.head()

In [ ]:
grader.check("q4")

<a id="q5"></a>

## Question 5: Set Difference — 12 Points

Implement `difference(df1, df2)` for identical schemas. Return complete rows in `df1` that do **not** occur in `df2`. Keep repeated unmatched rows. Preserve the first table's column order and data types.

Create `jfk_only_destinations`: destinations served from JFK but not LGA. This operation is directional.

Example: `[BOS, LAX] − [BOS, ATL]` gives `[LAX]`.

**Hint:** `pd.MultiIndex.from_frame` converts complete rows to comparable keys; its `isin` method can identify matching rows.

[Return to outline](#question-navigation)


In [ ]:
def difference(df1, df2):
    return ...

jfk_only_destinations = difference(jfk_routes, lga_routes)
jfk_only_destinations.head()

In [ ]:
grader.check("q5")

<a id="q6"></a>

## Question 6: Cross-product — 12 Points

Implement `cross_product(df1, df2)` to return every possible pair of input rows. Shared column names must receive pandas suffixes `_x` and `_y`.

Create `airline_origin_pairs` using `small_airlines` (3 rows) and `origins` (3 rows). The result should have **9 rows** and columns `carrier`, `name`, `origin`.

These are possible combinations, not evidence that each airline serves each airport. Use the small subsets; do not cross-join the full flights table.

**Hint:** `merge(..., how='cross')`.

[Return to outline](#question-navigation)


In [ ]:
def cross_product(df1, df2):
    return ...

airline_origin_pairs = cross_product(small_airlines, origins)
airline_origin_pairs

In [ ]:
grader.check("q6")

<a id="q7"></a>

## Question 7: Theta-Join — 10 Points

Implement `theta_join(df1, df2, condition)` using your `cross_product` and `select` functions. A theta-join keeps row pairs satisfying a condition.

Join `plane_sample` to itself, keeping pairs where `year_x < year_y`. The left aircraft was manufactured earlier than the right aircraft. Keep all cross-product columns, including the suffixes.

Example: years `[2000, 2010]` joined to themselves with `<` produce the pair `(2000, 2010)` only. Compare manufacturing years, not current ages.

[Return to outline](#question-navigation)


In [ ]:
def theta_join(df1, df2, condition):
    return ...

earlier_plane_pairs = theta_join(plane_sample, plane_sample, lambda row: row['year_x'] < row['year_y'])
earlier_plane_pairs

In [ ]:
grader.check("q7")

<a id="q8"></a>

## Question 8: Natural Join — 10 Points

Implement `natural_join(df1, df2)`.

- Match on **all column names shared by the two tables**.
- Return matching rows only, retaining **one copy** of each shared column.
- Keep all matching row pairs, with all left-table columns first, followed by right-only columns.
- If there are no shared columns, return the cross-product.

Create `named_flights` by naturally joining `flights.head(10)` with `airlines`. Their only shared column is `carrier`, so the result has **10 rows and 9 columns**, including one `carrier` column and the airline `name`.

**Important:** When joining `flights` and `planes`, both `tailnum` and `year` are common names, but the two `year` fields have different meanings. Rename conflicting attributes before applying a natural join. This question uses flights and airlines, where the shared key has the same meaning.

**Hint:** Find common column names, then use an inner `merge` on those columns.

[Return to outline](#question-navigation)


In [ ]:
def natural_join(df1, df2):
    return ...

named_flights = natural_join(flights.head(10), airlines)
named_flights

In [ ]:
grader.check("q8")

<a id="q9"></a>

## Question 9: Flights per Airline — 18 Points

Which airlines operated the most NYC departures in 2013? Complete these three steps using pandas directly. **Do not define a function.**

1. Create `flight_airlines` by inner-joining the full `flights` table with `airlines` on `carrier` using `merge`. Keep every flight record; do not remove repeated departures.
2. Create `airline_counts`, a Series counting flights by airline `name`. Sort from largest to smallest.
3. Create `top5_airlines`, the first five entries of `airline_counts`.

The joined table has **336,776 rows**. There are **16 airlines**. United Air Lines Inc. is first, with **58,665 flights**.

**Hints:** `merge`, `groupby('name').size()`, `sort_values(ascending=False)`, and `head(5)`.

[Return to outline](#question-navigation)


In [ ]:
flight_airlines = ...
airline_counts = ...
top5_airlines = ...

top5_airlines

In [ ]:
grader.check("q9")

## Congratulations! You have finished Lab09!

Run all cells in order and save your notebook. In the generated student notebook, use the export cell to create your submission ZIP and submit it to Gradescope.


## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)